In [32]:
import sqlite3
import pandas as pd
from datetime import datetime

# connecties met sdm en dwh databases (brondatabase + dwh database afgeleid van ETL-schema's)

# bron database
sdm_conn = sqlite3.connect("BikeToDriveDatabase.db")

# data warehouse
dwh_conn = sqlite3.connect("DWH_DB.db")

In [31]:
# als er iets mis is met de connection of foutjes in database run dit zodat de connection wordt gestopt en je opnieuw kan proberen !

sdm_conn.close()
dwh_conn.close()

In [ ]:
# dimensie klant scd type 1

sdm_klant = pd.read_sql("""
SELECT klantnr, naam, woonplaats, adres, geslacht, geboortedatum, 1 AS source_id
FROM Fiets_Verkoop_Klant

UNION ALL

SELECT klantnr, naam, woonplaats, adres, geslacht, geboortedatum, 2 AS source_id
FROM Accessoire_Verkoop_Klant
""", sdm_conn)

sdm_klant["business_key"] = (
    sdm_klant["naam"] + "_" +
    sdm_klant["geboortedatum"].astype(str)
)

sdm_klant = sdm_klant.drop_duplicates(subset=["business_key"])

dwh_klant = pd.read_sql("""
SELECT klant_key, business_key, source_id, klantnr, naam, woonplaats, adres, geslacht, geboortedatum
FROM Klant
""", dwh_conn)

merged = sdm_klant.merge(
    dwh_klant,
    on="business_key",
    how="left",
    suffixes=("_sdm", "_dwh")
)

for _, row in merged.iterrows():
    if pd.isna(row["klant_key"]):
        dwh_conn.execute("""
        INSERT INTO Klant (
            business_key, source_id, klantnr, naam, woonplaats, adres, geslacht, geboortedatum
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            row["business_key"],
            row["source_id_sdm"],
            row["klantnr_sdm"],
            row["naam_sdm"],
            row["woonplaats_sdm"],
            row["adres_sdm"],
            row["geslacht_sdm"],
            row["geboortedatum_sdm"]
        ))
    else:
        gewijzigd = (
            row["source_id_sdm"] != row["source_id_dwh"] or
            row["klantnr_sdm"] != row["klantnr_dwh"] or
            row["naam_sdm"] != row["naam_dwh"] or
            row["woonplaats_sdm"] != row["woonplaats_dwh"] or
            row["adres_sdm"] != row["adres_dwh"] or
            row["geslacht_sdm"] != row["geslacht_dwh"] or
            str(row["geboortedatum_sdm"]) != str(row["geboortedatum_dwh"])
        )

        if gewijzigd:
            dwh_conn.execute("""
            UPDATE Klant
            SET source_id = ?, klantnr = ?, naam = ?, woonplaats = ?, adres = ?, geslacht = ?, geboortedatum = ?
            WHERE klant_key = ?
            """, (
                row["source_id_sdm"],
                row["klantnr_sdm"],
                row["naam_sdm"],
                row["woonplaats_sdm"],
                row["adres_sdm"],
                row["geslacht_sdm"],
                row["geboortedatum_sdm"],
                int(row["klant_key"])
            ))

dwh_conn.commit()
print("Klant SCD1 klaar")

Klant SCD1 klaar


In [ ]:
# product scd type 1

sdm_product = pd.read_sql("""
SELECT 
    accessoirenr AS productnr,
    'accessoire' AS product_type,
    soort,
    naam AS merk,
    NULL AS type,
    NULL AS kleur,
    leverancier AS fabrikant
FROM Accessoire_Verkoop_Accessoire

UNION ALL

SELECT 
    accessoirenr AS productnr,
    'accessoire' AS product_type,
    soort,
    naam AS merk,
    NULL AS type,
    NULL AS kleur,
    leverancier AS fabrikant
FROM Accessoire_Inkoop_Accessoire

UNION ALL

SELECT 
    fietsnr AS productnr,
    'fiets' AS product_type,
    soort,
    merk,
    type,
    kleur,
    fabrikant
FROM Fiets_Verkoop_Fiets

UNION ALL

SELECT 
    fietsnr AS productnr,
    'fiets' AS product_type,
    soort,
    merk,
    type,
    kleur,
    fabrikant
FROM Fiets_Inkoop_Fiets
""", sdm_conn)

sdm_product["business_key"] = (
    sdm_product["product_type"] + "_" +
    sdm_product["productnr"].astype(str)
)

sdm_product = sdm_product.drop_duplicates(subset=["business_key"])

dwh_product = pd.read_sql("""
SELECT product_key, business_key, productnr, product_type, soort, merk, type, kleur, fabrikant
FROM Product
""", dwh_conn)

merged = sdm_product.merge(
    dwh_product,
    on="business_key",
    how="left",
    suffixes=("_sdm", "_dwh")
)

for _, row in merged.iterrows():
    if pd.isna(row["product_key"]):
        dwh_conn.execute("""
        INSERT INTO Product (
            business_key, productnr, product_type, soort, merk, type, kleur, fabrikant
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            row["business_key"],
            row["productnr_sdm"],
            row["product_type_sdm"],
            row["soort_sdm"],
            row["merk_sdm"],
            row["type_sdm"],
            row["kleur_sdm"],
            row["fabrikant_sdm"]
        ))
    else:
        gewijzigd = (
            row["productnr_sdm"] != row["productnr_dwh"] or
            row["product_type_sdm"] != row["product_type_dwh"] or
            row["soort_sdm"] != row["soort_dwh"] or
            str(row["merk_sdm"]) != str(row["merk_dwh"]) or
            str(row["type_sdm"]) != str(row["type_dwh"]) or
            str(row["kleur_sdm"]) != str(row["kleur_dwh"]) or
            str(row["fabrikant_sdm"]) != str(row["fabrikant_dwh"])
        )

        if gewijzigd:
            dwh_conn.execute("""
            UPDATE Product
            SET productnr = ?, product_type = ?, soort = ?, merk = ?, type = ?, kleur = ?, fabrikant = ?
            WHERE product_key = ?
            """, (
                row["productnr_sdm"],
                row["product_type_sdm"],
                row["soort_sdm"],
                row["merk_sdm"],
                row["type_sdm"],
                row["kleur_sdm"],
                row["fabrikant_sdm"],
                int(row["product_key"])
            ))

dwh_conn.commit()
print("Product SCD1 klaar")

Product SCD1 klaar


In [ ]:
# leverancier scd type 1

sdm_lev = pd.read_sql("""
SELECT leveranciernr, naam, adres, woonplaats
FROM Accessoire_Verkoop_Leverancier

UNION ALL

SELECT fabrikantnr AS leveranciernr, naam, adres, plaats AS woonplaats
FROM Fiets_Verkoop_Fabrikant
""", sdm_conn)

sdm_lev["business_key"] = (
    sdm_lev["naam"] + "_" +
    sdm_lev["adres"]
)

sdm_lev = sdm_lev.drop_duplicates(subset=["business_key"])

dwh_lev = pd.read_sql("""
SELECT lev_key, business_key, leveranciernr, naam, adres, woonplaats
FROM Leverancier
""", dwh_conn)

merged = sdm_lev.merge(
    dwh_lev,
    on="business_key",
    how="left",
    suffixes=("_sdm", "_dwh")
)

for _, row in merged.iterrows():
    if pd.isna(row["lev_key"]):
        dwh_conn.execute("""
        INSERT INTO Leverancier (
            business_key, leveranciernr, naam, adres, woonplaats
        )
        VALUES (?, ?, ?, ?, ?)
        """, (
            row["business_key"],
            row["leveranciernr_sdm"],
            row["naam_sdm"],
            row["adres_sdm"],
            row["woonplaats_sdm"]
        ))
    else:
        gewijzigd = (
            row["leveranciernr_sdm"] != row["leveranciernr_dwh"] or
            row["naam_sdm"] != row["naam_dwh"] or
            row["adres_sdm"] != row["adres_dwh"] or
            row["woonplaats_sdm"] != row["woonplaats_dwh"]
        )

        if gewijzigd:
            dwh_conn.execute("""
            UPDATE Leverancier
            SET leveranciernr = ?, naam = ?, adres = ?, woonplaats = ?
            WHERE lev_key = ?
            """, (
                row["leveranciernr_sdm"],
                row["naam_sdm"],
                row["adres_sdm"],
                row["woonplaats_sdm"],
                int(row["lev_key"])
            ))

dwh_conn.commit()
print("Leverancier SCD1 klaar")

Leverancier SCD1 klaar


In [ ]:
# datum (geen scd type nodig)

sdm_datum = pd.read_sql("""
SELECT datum FROM Fiets_Verkoop
UNION ALL
SELECT datum FROM Accessoire_Verkoop
""", sdm_conn)

sdm_datum = sdm_datum.drop_duplicates()
sdm_datum["datum"] = pd.to_datetime(sdm_datum["datum"])
sdm_datum["dag"] = sdm_datum["datum"].dt.day
sdm_datum["maand"] = sdm_datum["datum"].dt.month
sdm_datum["jaar"] = sdm_datum["datum"].dt.year
sdm_datum["kwartaal"] = sdm_datum["datum"].dt.quarter
sdm_datum["datum_key"] = sdm_datum["datum"].dt.strftime("%Y%m%d").astype(int)

dwh_keys = pd.read_sql("SELECT datum_key FROM Datum", dwh_conn)

nieuwe_datums = sdm_datum[
    ~sdm_datum["datum_key"].isin(dwh_keys["datum_key"])
]

for _, row in nieuwe_datums.iterrows():
    dwh_conn.execute("""
    INSERT INTO Datum (datum_key, dag, maand, kwartaal, jaar)
    VALUES (?, ?, ?, ?, ?)
    """, (
        int(row["datum_key"]),
        int(row["dag"]),
        int(row["maand"]),
        int(row["kwartaal"]),
        int(row["jaar"])
    ))

dwh_conn.commit()
print("Datum geladen")

Datum geladen


In [ ]:
# inkoopperiode (geen scd type nodig)

sdm_inkoopperiode = pd.read_sql("""
SELECT inkoopmaand, inkoopjaar
FROM Fiets_Inkoop

UNION ALL

SELECT inkoopmaand, inkoopjaar
FROM Accessoire_Inkoop
""", sdm_conn)

sdm_inkoopperiode = sdm_inkoopperiode.drop_duplicates()
sdm_inkoopperiode = sdm_inkoopperiode.sort_values(
    by=["inkoopjaar", "inkoopmaand"]
).reset_index(drop=True)

dwh_periodes = pd.read_sql("""
SELECT inkoopmaand, inkoopjaar
FROM InkoopPeriode
""", dwh_conn)

nieuwe_periodes = sdm_inkoopperiode.merge(
    dwh_periodes,
    on=["inkoopmaand", "inkoopjaar"],
    how="left",
    indicator=True
).query('_merge == "left_only"').drop(columns=["_merge"])

start_key = pd.read_sql("""
SELECT COALESCE(MAX(periode_key), 0) AS max_key
FROM InkoopPeriode
""", dwh_conn)["max_key"][0]

nieuwe_periodes["periode_key"] = range(
    start_key + 1,
    start_key + 1 + len(nieuwe_periodes)
)

nieuwe_periodes = nieuwe_periodes[
    ["periode_key", "inkoopmaand", "inkoopjaar"]
]

nieuwe_periodes.to_sql(
    "InkoopPeriode",
    dwh_conn,
    if_exists="append",
    index=False
)

print("InkoopPeriode geladen")

InkoopPeriode geladen


In [ ]:
# monteur scd type 2

etl_tijd = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

sdm_monteur = pd.read_sql("""
SELECT monteurnr, naam, woonplaats, uurloon
FROM Accessoire_Verkoop_Monteur

UNION ALL

SELECT monteurnr, naam, woonplaats, uurloon
FROM Fiets_Verkoop_Monteur
""", sdm_conn)

sdm_monteur["business_key"] = (
    sdm_monteur["naam"] + "_" +
    sdm_monteur["woonplaats"]
)

sdm_monteur = sdm_monteur.drop_duplicates(subset=["business_key"])

dwh_monteur = pd.read_sql("""
SELECT *
FROM Monteur
WHERE eind_tijd IS NULL
""", dwh_conn)

merged = sdm_monteur.merge(
    dwh_monteur,
    on="business_key",
    how="left",
    suffixes=("_sdm", "_dwh")
)

for _, row in merged.iterrows():
    if pd.isna(row["monteur_key"]):
        dwh_conn.execute("""
        INSERT INTO Monteur (
            business_key, monteurnr, naam, woonplaats, uurloon, begin_tijd, eind_tijd
        )
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            row["business_key"],
            row["monteurnr_sdm"],
            row["naam_sdm"],
            row["woonplaats_sdm"],
            row["uurloon_sdm"],
            etl_tijd,
            None
        ))
    else:
        gewijzigd = (
            row["monteurnr_sdm"] != row["monteurnr_dwh"] or
            row["naam_sdm"] != row["naam_dwh"] or
            row["woonplaats_sdm"] != row["woonplaats_dwh"] or
            row["uurloon_sdm"] != row["uurloon_dwh"]
        )

        if gewijzigd:
            dwh_conn.execute("""
            UPDATE Monteur
            SET eind_tijd = ?
            WHERE monteur_key = ?
            """, (
                etl_tijd,
                int(row["monteur_key"])
            ))

            dwh_conn.execute("""
            INSERT INTO Monteur (
                business_key, monteurnr, naam, woonplaats, uurloon, begin_tijd, eind_tijd
            )
            VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (
                row["business_key"],
                row["monteurnr_sdm"],
                row["naam_sdm"],
                row["woonplaats_sdm"],
                row["uurloon_sdm"],
                etl_tijd,
                None
            ))

dwh_conn.commit()
print("Monteur SCD2 klaar")

Monteur SCD2 klaar


In [ ]:
# filiaal scd type 2

etl_tijd = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

sdm_filiaal = pd.read_sql("""
SELECT filiaalnr, naam, adres, provincie
FROM Accessoire_Verkoop_Filiaal

UNION ALL

SELECT filiaalnr, naam, adres, provincie
FROM Fiets_Verkoop_Filiaal
""", sdm_conn)

sdm_filiaal["business_key"] = sdm_filiaal["adres"]
sdm_filiaal = sdm_filiaal.drop_duplicates(subset=["business_key"])

dwh_filiaal = pd.read_sql("""
SELECT *
FROM Filiaal
WHERE eind_tijd IS NULL
""", dwh_conn)

merged = sdm_filiaal.merge(
    dwh_filiaal,
    on="business_key",
    how="left",
    suffixes=("_sdm", "_dwh")
)

for _, row in merged.iterrows():
    if pd.isna(row["filiaal_key"]):
        dwh_conn.execute("""
        INSERT INTO Filiaal (
            business_key, filiaalnr, naam, adres, provincie, begin_tijd, eind_tijd
        )
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            row["business_key"],
            row["filiaalnr_sdm"],
            row["naam_sdm"],
            row["adres_sdm"],
            row["provincie_sdm"],
            etl_tijd,
            None
        ))
    else:
        gewijzigd = (
            row["filiaalnr_sdm"] != row["filiaalnr_dwh"] or
            row["naam_sdm"] != row["naam_dwh"] or
            row["adres_sdm"] != row["adres_dwh"] or
            row["provincie_sdm"] != row["provincie_dwh"]
        )

        if gewijzigd:
            dwh_conn.execute("""
            UPDATE Filiaal
            SET eind_tijd = ?
            WHERE filiaal_key = ?
            """, (
                etl_tijd,
                int(row["filiaal_key"])
            ))

            dwh_conn.execute("""
            INSERT INTO Filiaal (
                business_key, filiaalnr, naam, adres, provincie, begin_tijd, eind_tijd
            )
            VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (
                row["business_key"],
                row["filiaalnr_sdm"],
                row["naam_sdm"],
                row["adres_sdm"],
                row["provincie_sdm"],
                etl_tijd,
                None
            ))

dwh_conn.commit()
print("Filiaal SCD2 klaar")

Filiaal SCD2 klaar


In [ ]:
# feittabel verkoop
# incremental loading

sdm_verkoop = pd.read_sql("""
    SELECT
        fv.fiets_verkoopnr AS verkoopnr,
        1 AS source_id,
        fv.datum,
        fv.aantal,
        fv.verkoopprijs,
        fv.klant,
        fv.fiets AS productnr,
        'fiets' AS product_type,
        fv.monteur,
        k.naam AS klant_naam,
        k.geboortedatum,
        m.naam AS monteur_naam,
        m.woonplaats AS monteur_woonplaats,
        f.adres AS filiaal_adres
    FROM Fiets_Verkoop fv
    JOIN Fiets_Verkoop_Klant k ON fv.klant = k.klantnr
    JOIN Fiets_Verkoop_Monteur m ON fv.monteur = m.monteurnr
    JOIN Fiets_Verkoop_Filiaal f ON m.filiaal = f.filiaalnr

    UNION ALL

    SELECT
        av.accessoire_verkoopnr AS verkoopnr,
        2 AS source_id,
        av.datum,
        av.aantal,
        av.verkoopprijs,
        av.klant,
        av.accessoire AS productnr,
        'accessoire' AS product_type,
        av.monteur,
        k.naam AS klant_naam,
        k.geboortedatum,
        m.naam AS monteur_naam,
        m.woonplaats AS monteur_woonplaats,
        f.adres AS filiaal_adres
    FROM Accessoire_Verkoop av
    JOIN Accessoire_Verkoop_Klant k ON av.klant = k.klantnr
    JOIN Accessoire_Verkoop_Monteur m ON av.monteur = m.monteurnr
    JOIN Accessoire_Verkoop_Filiaal f ON m.filiaal = f.filiaalnr
""", sdm_conn)

sdm_verkoop["datum"] = pd.to_datetime(sdm_verkoop["datum"])
sdm_verkoop["datum_key"] = sdm_verkoop["datum"].dt.strftime("%Y%m%d").astype(int)

sdm_verkoop["klant_business_key"] = (
    sdm_verkoop["klant_naam"] + "_" +
    sdm_verkoop["geboortedatum"].astype(str)
)

sdm_verkoop["monteur_business_key"] = (
    sdm_verkoop["monteur_naam"] + "_" +
    sdm_verkoop["monteur_woonplaats"]
)

sdm_verkoop["product_business_key"] = (
    sdm_verkoop["product_type"] + "_" +
    sdm_verkoop["productnr"].astype(str)
)

sdm_verkoop["filiaal_business_key"] = sdm_verkoop["filiaal_adres"]

klant_dim = pd.read_sql("SELECT klant_key, business_key FROM Klant", dwh_conn)
product_dim = pd.read_sql("SELECT product_key, business_key FROM Product", dwh_conn)
monteur_dim = pd.read_sql("SELECT monteur_key, business_key FROM Monteur WHERE eind_tijd IS NULL", dwh_conn)
filiaal_dim = pd.read_sql("SELECT filiaal_key, business_key FROM Filiaal WHERE eind_tijd IS NULL", dwh_conn)
datum_dim = pd.read_sql("SELECT datum_key FROM Datum", dwh_conn)

sdm_verkoop = sdm_verkoop.merge(
    klant_dim, left_on="klant_business_key", right_on="business_key", how="left"
).drop(columns=["business_key"])

sdm_verkoop = sdm_verkoop.merge(
    monteur_dim, left_on="monteur_business_key", right_on="business_key", how="left"
).drop(columns=["business_key"])

sdm_verkoop = sdm_verkoop.merge(
    product_dim, left_on="product_business_key", right_on="business_key", how="left"
).drop(columns=["business_key"])

sdm_verkoop = sdm_verkoop.merge(
    filiaal_dim, left_on="filiaal_business_key", right_on="business_key", how="left"
).drop(columns=["business_key"])

sdm_verkoop = sdm_verkoop.merge(
    datum_dim, on="datum_key", how="left"
)

fact_verkoop = sdm_verkoop[
    ["source_id", "verkoopnr", "klant_key", "monteur_key", "product_key", "filiaal_key", "datum_key", "verkoopprijs", "aantal"]
].copy()

fact_verkoop = fact_verkoop.dropna(
    subset=["klant_key", "monteur_key", "product_key", "filiaal_key", "datum_key"]
)

fact_verkoop["source_id"] = fact_verkoop["source_id"].astype(int)
fact_verkoop["verkoopnr"] = fact_verkoop["verkoopnr"].astype(int)
fact_verkoop["klant_key"] = fact_verkoop["klant_key"].astype(int)
fact_verkoop["monteur_key"] = fact_verkoop["monteur_key"].astype(int)
fact_verkoop["product_key"] = fact_verkoop["product_key"].astype(int)
fact_verkoop["filiaal_key"] = fact_verkoop["filiaal_key"].astype(int)
fact_verkoop["datum_key"] = fact_verkoop["datum_key"].astype(int)

bestaande_verkoop = pd.read_sql("""
SELECT source_id, verkoopnr
FROM Verkoop
""", dwh_conn)

nieuwe_fact_verkoop = fact_verkoop.merge(
    bestaande_verkoop,
    on=["source_id", "verkoopnr"],
    how="left",
    indicator=True
).query('_merge == "left_only"').drop(columns=["_merge"])

nieuwe_fact_verkoop.to_sql(
    "Verkoop",
    dwh_conn,
    if_exists="append",
    index=False
)

print(nieuwe_fact_verkoop.head())

     source_id  verkoopnr  klant_key  monteur_key  product_key  filiaal_key  \
150          1       9999          1           11           14            1   

     datum_key  verkoopprijs  aantal  
150   20240101         500.0       1  


In [ ]:
#feittabel onderhoud
# incremental loading

sdm_onderhoud = pd.read_sql("""
    SELECT
        o.onderhoudnr,
        o.datum,
        o.starttijd,
        o.eindtijd,
        o.fiets AS productnr,
        'fiets' AS product_type,
        o.monteur,
        m.naam AS monteur_naam,
        m.woonplaats AS monteur_woonplaats,
        m.uurloon,
        m.filiaal,
        f.adres AS filiaal_adres
    FROM Onderhoud o
    JOIN Onderhoud_Monteur m ON o.monteur = m.monteurnr
    JOIN Onderhoud_Filiaal f ON m.filiaal = f.filiaalnr
""", sdm_conn)

sdm_onderhoud["product_business_key"] = (
    sdm_onderhoud["product_type"] + "_" +
    sdm_onderhoud["productnr"].astype(str)
)

sdm_onderhoud["monteur_business_key"] = (
    sdm_onderhoud["monteur_naam"] + "_" +
    sdm_onderhoud["monteur_woonplaats"]
)

sdm_onderhoud["filiaal_business_key"] = sdm_onderhoud["filiaal_adres"]

product_dim = pd.read_sql("SELECT product_key, business_key FROM Product", dwh_conn)
monteur_dim = pd.read_sql("SELECT monteur_key, business_key FROM Monteur WHERE eind_tijd IS NULL", dwh_conn)
filiaal_dim = pd.read_sql("SELECT filiaal_key, business_key FROM Filiaal WHERE eind_tijd IS NULL", dwh_conn)

sdm_onderhoud = sdm_onderhoud.merge(
    product_dim, left_on="product_business_key", right_on="business_key", how="left"
).drop(columns=["business_key"])

sdm_onderhoud = sdm_onderhoud.merge(
    monteur_dim, left_on="monteur_business_key", right_on="business_key", how="left"
).drop(columns=["business_key"])

sdm_onderhoud = sdm_onderhoud.merge(
    filiaal_dim, left_on="filiaal_business_key", right_on="business_key", how="left"
).drop(columns=["business_key"])

fact_onderhoud = sdm_onderhoud[
    ["onderhoudnr", "product_key", "monteur_key", "filiaal_key", "datum", "starttijd", "eindtijd", "uurloon"]
].copy()

fact_onderhoud = fact_onderhoud.dropna(
    subset=["product_key", "monteur_key", "filiaal_key"]
)

fact_onderhoud["onderhoudnr"] = fact_onderhoud["onderhoudnr"].astype(int)
fact_onderhoud["product_key"] = fact_onderhoud["product_key"].astype(int)
fact_onderhoud["monteur_key"] = fact_onderhoud["monteur_key"].astype(int)
fact_onderhoud["filiaal_key"] = fact_onderhoud["filiaal_key"].astype(int)

bestaande_onderhoud = pd.read_sql("""
SELECT onderhoudnr
FROM Onderhoud
""", dwh_conn)

nieuwe_fact_onderhoud = fact_onderhoud[
    ~fact_onderhoud["onderhoudnr"].isin(bestaande_onderhoud["onderhoudnr"])
]

nieuwe_fact_onderhoud.to_sql(
    "Onderhoud",
    dwh_conn,
    if_exists="append",
    index=False
)

print(nieuwe_fact_onderhoud.head())

   onderhoudnr  product_key  monteur_key  filiaal_key       datum  \
1            2           31            8            4  2024-02-05   
3            4           24            2            1  2024-11-16   
6            7           17            8            4  2024-05-14   
7            8           80            3            1  2024-10-10   
9           10           46            2            1  2024-08-21   

          starttijd          eindtijd  uurloon  
1  14:00:00.0000000  15:00:00.0000000    16.95  
3  10:00:00.0000000  11:00:00.0000000    18.75  
6  11:00:00.0000000  12:00:00.0000000    16.95  
7  09:00:00.0000000  10:00:00.0000000    20.00  
9  08:00:00.0000000  09:00:00.0000000    18.75  


In [ ]:
# feittabel inkoop incremental loading

sdm_inkoop = pd.read_sql("""
    SELECT 
        ai.inkoopnr,
        1 AS source_id,
        ai.inkoopmaand,
        ai.inkoopjaar,
        ai.aantal,
        a.accessoirenr AS productnr,
        'accessoire' AS product_type,
        l.leveranciernr,
        l.naam AS leverancier_naam,
        l.adres AS leverancier_adres,
        a.inkoopprijs
    FROM Accessoire_Inkoop ai
    JOIN Accessoire_Inkoop_Accessoire a 
        ON ai.accessoire = a.accessoirenr
    JOIN Accessoire_Inkoop_Leverancier l 
        ON a.leverancier = l.leveranciernr

    UNION ALL

    SELECT 
        fi.inkoopnr,
        2 AS source_id,
        fi.inkoopmaand,
        fi.inkoopjaar,
        fi.aantal,
        f.fietsnr AS productnr,
        'fiets' AS product_type,
        fab.fabrikantnr AS leveranciernr,
        fab.naam AS leverancier_naam,
        fab.adres AS leverancier_adres,
        f.inkoopprijs
    FROM Fiets_Inkoop fi
    JOIN Fiets_Inkoop_Fiets f 
        ON fi.fiets = f.fietsnr
    JOIN Fiets_Inkoop_Fabrikant fab 
        ON f.fabrikant = fab.fabrikantnr
""", sdm_conn)

sdm_inkoop["product_business_key"] = (
    sdm_inkoop["product_type"] + "_" +
    sdm_inkoop["productnr"].astype(str)
)

sdm_inkoop["lev_business_key"] = (
    sdm_inkoop["leverancier_naam"] + "_" +
    sdm_inkoop["leverancier_adres"]
)

product_dim = pd.read_sql("SELECT product_key, business_key FROM Product", dwh_conn)
periode_dim = pd.read_sql("SELECT periode_key, inkoopmaand, inkoopjaar FROM InkoopPeriode", dwh_conn)
leverancier_dim = pd.read_sql("SELECT lev_key, business_key FROM Leverancier", dwh_conn)

sdm_inkoop = sdm_inkoop.merge(
    product_dim,
    left_on="product_business_key",
    right_on="business_key",
    how="left"
).drop(columns=["business_key"])

sdm_inkoop = sdm_inkoop.merge(
    periode_dim,
    on=["inkoopmaand", "inkoopjaar"],
    how="left"
)

sdm_inkoop = sdm_inkoop.merge(
    leverancier_dim,
    left_on="lev_business_key",
    right_on="business_key",
    how="left"
).drop(columns=["business_key"])

fact_inkoop = sdm_inkoop[
    ["source_id", "inkoopnr", "product_key", "periode_key", "lev_key", "aantal", "inkoopprijs"]
].copy()

fact_inkoop = fact_inkoop.dropna(subset=["product_key", "periode_key", "lev_key"])

fact_inkoop["source_id"] = fact_inkoop["source_id"].astype(int)
fact_inkoop["inkoopnr"] = fact_inkoop["inkoopnr"].astype(int)
fact_inkoop["product_key"] = fact_inkoop["product_key"].astype(int)
fact_inkoop["periode_key"] = fact_inkoop["periode_key"].astype(int)
fact_inkoop["lev_key"] = fact_inkoop["lev_key"].astype(int)

bestaande_inkoop = pd.read_sql("""
SELECT source_id, inkoopnr
FROM Inkoop
""", dwh_conn)

nieuwe_fact_inkoop = fact_inkoop.merge(
    bestaande_inkoop,
    on=["source_id", "inkoopnr"],
    how="left",
    indicator=True
).query('_merge == "left_only"').drop(columns=["_merge"])

nieuwe_fact_inkoop.to_sql(
    "Inkoop",
    dwh_conn,
    if_exists="append",
    index=False
)

print(nieuwe_fact_inkoop.head())

   source_id  inkoopnr  product_key  periode_key  lev_key  aantal  inkoopprijs
0          1         1            7            4        2      38        24.90
1          1         2            6            8        2      31        18.75
2          1         3            2            1        1      41         7.25
3          1         4            1            1        1      21         8.50
4          1         5            4           10        1      31        11.40


In [42]:
# scd type 1 TEST KLANT
# woonplaats veranderd voor jam jansen van amsterdam > leidschendam
pd.read_sql("SELECT * FROM Fiets_Verkoop_Klant LIMIT 5", sdm_conn)

sdm_conn.execute("""
UPDATE Fiets_Verkoop_Klant
SET woonplaats = 'Amsterdam'
WHERE klantnr = 1
""")

sdm_conn.commit()

In [45]:
# scd type 1 TEST KLANT
# simpele select query om te laten zien dat t aangepast is.
pd.read_sql("""
SELECT klant_key, business_key, naam, woonplaats, adres, geboortedatum
FROM Klant
LIMIT 10
""", dwh_conn)

,klant_key,business_key,naam,woonplaats,adres,geboortedatum
0,1,Jan Jansen_1985-03-22,Jan Jansen,Amsterdam,Kerkstraat 12,1985-03-22
1,2,Sophie de Boer_1990-07-11,Sophie de Boer,Rotterdam,Lindelaan 8,1990-07-11
2,3,Pieter Visser_1978-11-05,Pieter Visser,Den Haag,Havenstraat 3,1978-11-05
3,4,Emma Smit_1995-02-18,Emma Smit,Haarlem,Boomgaard 22,1995-02-18
4,5,Tom Bakker_1982-09-09,Tom Bakker,Leiden,Stationsweg 44,1982-09-09
5,6,Lisa Meijer_1993-12-30,Lisa Meijer,Zaandam,Dijkstraat 10,1993-12-30
6,7,Bart de Vries_1976-06-06,Bart de Vries,Delft,Brouwersgracht 7,1976-06-06
7,8,Julia van Dijk_2000-01-15,Julia van Dijk,Hoorn,Plataanlaan 5,2000-01-15
8,9,Kevin Mol_1989-08-01,Kevin Mol,Alkmaar,Singel 99,1989-08-01
9,10,Nina Groen_1991-05-25,Nina Groen,Schiedam,Waterstraat 16,1991-05-25


In [55]:
# scd type 2 TEST MONTEUR
# voor monteur tom van dijk (monteurnr = 1) uurloon aangepast 19.5 > 119.5 (he gettin rich fr)

pd.read_sql("SELECT * FROM Fiets_Verkoop_Monteur LIMIT 5", sdm_conn)

sdm_conn.execute("""
UPDATE Fiets_Verkoop_Monteur
SET uurloon = 119.5
WHERE monteurnr = 1
""")

sdm_conn.execute("""
UPDATE Accessoire_Verkoop_Monteur
SET uurloon = 119.5
WHERE monteurnr = 1
""")
sdm_conn.commit()

In [59]:
# scd type 2 TEST MONTEUR 
# select query om aan te tonen (kan ook gw in database kijken!)
pd.read_sql("""
SELECT *
FROM Fiets_Verkoop_Monteur
WHERE monteurnr = 1
""", sdm_conn)

,monteurnr,naam,woonplaats,uurloon,filiaal
0,1,Tom van Dijk,Amsterdam,119.5,1


In [60]:
# FEITTABEL VERKOOP TEST !! 
# nieuwe verkoop toevoegen en kijken of t aanpast slay queen purr

sdm_conn.execute("""
INSERT INTO Fiets_Verkoop (
    fiets_verkoopnr, datum, aantal, verkoopprijs, klant, fiets, monteur
)
VALUES (9999, '2024-01-01', 1, 500, 1, 1, 1)
""")

sdm_conn.commit()